# Aula 5 — Séries Temporais com Prophet

Como usar este notebook: a **Parte A** tem células que o professor roda e
explica durante a aula. Acompanhe na tela, sem precisar digitar nada. A
**Parte B** é com você: complete os exercícios nos lugares marcados com
`# SEU CODIGO AQUI`.

Esta é a única aula do curso que instala uma biblioteca. A primeira
célula demora cerca de um minuto no Colab. Rode ela primeiro e espere
terminar antes de continuar.

## Parte A: Demonstração

In [ ]:
!pip install prophet -q

### Os dados e o formato que o Prophet exige

O Prophet só aceita duas colunas, com estes nomes exatos:

| Coluna | O que é |
|---|---|
| `ds` | a data |
| `y` | o valor que você quer prever |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet

# Endereço dos dados desta aula no GitHub.
URL_DADOS = "https://raw.githubusercontent.com/klein-natan/nanodegree-AI-Atitus/main/data/vendas_cafeteria.csv"
# Alternativa para testar offline, antes do repositório existir no GitHub:
# URL_DADOS = "../../data/vendas_cafeteria.csv"

dados = pd.read_csv(URL_DADOS, parse_dates=["data"])
dados = dados.sort_values("data").reset_index(drop=True)

# O Prophet exige os nomes ds e y: renomear é sempre o primeiro passo
serie = dados.rename(columns={"data": "ds", "vendas": "y"})[["ds", "y"]]
serie.head()

In [ ]:
# O mesmo corte no tempo da aula passada: os últimos 28 dias ficam escondidos
DIAS_DE_TESTE = 28

treino = serie.iloc[:-DIAS_DE_TESTE]
teste = serie.iloc[-DIAS_DE_TESTE:]

print(f"Treino: {len(treino)} dias, até {treino['ds'].max().date()}")
print(f"Teste:  {len(teste)} dias, a partir de {teste['ds'].min().date()}")

### Treinar e prever

O modelo do Prophet é a soma das peças que você já conhece:

$$y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

Onde $g(t)$ é a tendência, $s(t)$ são as sazonalidades, $h(t)$ são os
feriados e $\varepsilon_t$ é o que sobra.

In [ ]:
modelo = Prophet()
modelo.fit(treino)

futuro = modelo.make_future_dataframe(periods=DIAS_DE_TESTE)
previsao = modelo.predict(futuro)

# A saída tem uma linha por dia, com a previsão e as bordas do intervalo
previsao[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

In [ ]:
figura = modelo.plot(previsao)
plt.title("Vendas observadas e previsão do Prophet")
plt.xlabel("Data")
plt.ylabel("Vendas (R$)")
plt.show()

### As peças que o modelo estimou sozinho

Este é o gráfico que vale a reunião: cada peça do modelo, separada.

In [ ]:
figura_componentes = modelo.plot_components(previsao)
plt.show()

### O teste que importa: bateu a régua?

A régua da Aula 4 é a previsão sazonal ingênua, que copia a última
semana e erra R\\$ 63,78 por dia.

In [ ]:
def calcular_metricas(reais, previstos):
    # As mesmas tres metricas da Aula 4
    erro = np.asarray(reais) - np.asarray(previstos)
    mae = np.mean(np.abs(erro))
    rmse = np.sqrt(np.mean(erro ** 2))
    mape = np.mean(np.abs(erro / np.asarray(reais))) * 100
    return mae, rmse, mape

# Só as últimas 28 linhas da previsão correspondem ao teste
previsao_teste = previsao.iloc[-DIAS_DE_TESTE:]["yhat"].to_numpy()

# A régua: repetir a última semana do treino
ultima_semana = treino["y"].iloc[-7:].to_numpy()
sazonal_ingenua = np.tile(ultima_semana, 4)[:DIAS_DE_TESTE]

for nome, previsto in [("Sazonal ingênua", sazonal_ingenua),
                       ("Prophet", previsao_teste)]:
    mae, rmse, mape = calcular_metricas(teste["y"], previsto)
    print(f"{nome:<18} MAE R$ {mae:6.2f} | RMSE R$ {rmse:6.2f} | MAPE {mape:5.2f}%")

### O que o modelo não adivinha: o calendário

O Prophet descobre tendência e sazonalidade sozinho. Feriado, não: ele
não tem como saber que 25 de dezembro é diferente. A lista de datas
especiais é conhecimento de negócio, e quem entrega é você.

In [ ]:
# A tabela de feriados: uma linha por data, com um nome de grupo
feriados = pd.DataFrame({
    "holiday": "feriado_nacional",
    "ds": dados[dados["feriado"] == 1]["data"],
})

modelo_com_feriados = Prophet(holidays=feriados)
modelo_com_feriados.fit(treino)

futuro_2 = modelo_com_feriados.make_future_dataframe(periods=DIAS_DE_TESTE)
previsao_2 = modelo_com_feriados.predict(futuro_2)
previsao_teste_2 = previsao_2.iloc[-DIAS_DE_TESTE:]["yhat"].to_numpy()

mae, rmse, mape = calcular_metricas(teste["y"], previsao_teste_2)
print(f"Prophet com feriados: MAE R$ {mae:.2f} | RMSE R$ {rmse:.2f} | MAPE {mape:.2f}%")

## Parte B: Exercícios

Complete cada exercício no espaço marcado com `# SEU CODIGO AQUI`. Rode a
célula de verificação logo depois para conferir sua resposta.

### Exercício 1: conferindo o formato

Rode a célula e confirme que a tabela `serie` tem só as colunas `ds` e
`y`, e que a última data é 31 de dezembro de 2024.

In [ ]:
print(serie.columns.tolist())
print(serie.tail(3))

In [ ]:
if list(serie.columns) == ["ds", "y"]:
    print("✅ A tabela está no formato que o Prophet exige.")
else:
    print("❌ O Prophet só aceita as colunas ds e y, com esses nomes exatos.")

### Exercício 2: treinando o seu modelo

Crie um `Prophet()` chamado `meu_modelo` e treine com a tabela `treino`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print("Modelo treinado com", len(treino), "dias de histórico.")

In [ ]:
if hasattr(meu_modelo, "params"):
    print("✅ O modelo foi treinado.")
else:
    print("❌ Faltou chamar o fit com a tabela de treino.")

### Exercício 3: prevendo 28 dias

Crie a tabela de datas futuras com `make_future_dataframe(periods=28)` e
gere a previsão com `predict`.

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(minha_previsao[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(3))

In [ ]:
if len(minha_previsao) == len(treino) + 28:
    print("✅ A previsão cobre o histórico inteiro mais os 28 dias novos.")
else:
    print("❌ Confira o periods=28 no make_future_dataframe.")

### Exercício 4: comparando com a régua

Pegue só as últimas 28 linhas de `minha_previsao` e calcule o MAE contra
o `teste`. Compare com os R\\$ 63,78 da sazonal ingênua.

In [ ]:
reais = teste["y"].to_numpy()

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"MAE do seu Prophet: R$ {meu_mae:.2f}")
print(f"MAE da régua:       R$ 63,78")

In [ ]:
if 40 < meu_mae < 80:
    print("✅ Erro na faixa esperada. Repare que ainda não vencemos a régua com folga.")
else:
    print("❌ Confira se você usou só as últimas 28 linhas da previsão.")

### Exercício 5: ensinando o calendário

Monte a tabela de feriados e treine um novo modelo com
`Prophet(holidays=meus_feriados)`. Depois calcule o MAE de novo.

In [ ]:
datas_de_feriado = dados[dados["feriado"] == 1]["data"]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
mae_feriados, _, _ = calcular_metricas(
    reais, previsao_feriados.iloc[-28:]["yhat"].to_numpy()
)
print(f"MAE sem feriados: R$ {meu_mae:.2f}")
print(f"MAE com feriados: R$ {mae_feriados:.2f}")

In [ ]:
if mae_feriados < meu_mae:
    print("✅ A lista de feriados melhorou a previsão. O calendário é conhecimento seu, não do modelo.")
else:
    print("❌ Esperava uma melhora. Confira se a tabela tem as colunas holiday e ds.")

### Exercício 6: lendo o intervalo de incerteza

Conte quantos dias do teste caíram dentro da faixa entre `yhat_lower` e
`yhat_upper`. A faixa promete cobrir 80% dos casos.

In [ ]:
faixa = previsao_feriados.iloc[-28:]

In [ ]:
# SEU CODIGO AQUI

In [ ]:
print(f"Dias dentro da faixa: {dentro_da_faixa.sum()} de 28 ({cobertura:.0f}%)")
print(f"Largura média da faixa: R$ {(faixa['yhat_upper'] - faixa['yhat_lower']).mean():.2f}")

In [ ]:
if 50 <= cobertura <= 100:
    print("✅ Cobertura na faixa esperada. Lembre: 80% é promessa, não garantia.")
else:
    print("❌ Confira se você comparou os valores reais com yhat_lower e yhat_upper.")

### Exercício 7: desafio, prever 90 dias

Refaça a previsão para 90 dias à frente e meça a largura da faixa de
incerteza no primeiro e no último dia previsto. Ela abre muito?

In [ ]:
DIAS_LONGOS = 90

In [ ]:
# SEU CODIGO AQUI

In [ ]:
ultimos = previsao_longa.iloc[-DIAS_LONGOS:]
largura = ultimos["yhat_upper"] - ultimos["yhat_lower"]
print(f"Largura da faixa no 1º dia previsto:  R$ {largura.iloc[0]:.2f}")
print(f"Largura da faixa no 90º dia previsto: R$ {largura.iloc[-1]:.2f}")

plt.plot(ultimos["ds"], largura)
plt.xlabel("Data")
plt.ylabel("Largura do intervalo (R$)")
plt.title("A incerteza cresce com o horizonte")
plt.show()

In [ ]:
if len(largura) == DIAS_LONGOS:
    print("✅ Você mediu a faixa nos 90 dias.")
    print("   Repare que ela quase não abre: nesta série, o barulho do dia a dia domina,")
    print("   e a tendência é regular demais para gerar dúvida. Numa série instável, abriria.")
else:
    print("❌ Esperava 90 dias de previsão. Confira o periods=90.")

Agora, em texto: a dona da cafeteria quer saber quanto vai vender daqui a
três meses para decidir se contrata mais gente. Em duas ou três frases,
escreva o que você responderia, usando a previsão, a faixa de incerteza e
o que o modelo não tem como saber.
Edite esta célula (duplo clique nela) e escreva sua resposta no lugar
deste parágrafo.